In [16]:
# Imports
import os
import pandas as pd
from datasets import Dataset
import json
import traceback
from typing import List, Dict # Utile pour les type hints
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

C:\Users\Iwiko\AppData\Local\Temp\ipykernel_13436\3581978192.py:11: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\Iwiko\AppData\Local\Temp\ipykernel_13436\3581978192.py:11: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\Iwiko\AppData\Local\Temp\ipykernel_13436\3581978192.py:11: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\Iwiko\AppData\Local\Temp\ipykernel

In [17]:
with open("rag_predictions.json", encoding="utf-8") as f:
    evaluation_data = json.load(f)

evaluation_dataset = Dataset.from_dict({
    "user_input": evaluation_data["questions"],
    "response": evaluation_data["answers"],
    "retrieved_contexts": evaluation_data["contexts"],
    "reference": evaluation_data["ground_truths"],
})
print("Dataset d'évaluation prêt.")

Dataset d'évaluation prêt.


In [18]:
load_dotenv()

try:
    # 1. LLM juge : ChatOpenAI (evaluate → LangchainLLMWrapper, n=3 pour answer_relevancy)
    print("Initialisation LLM et Embeddings OpenAI...")
    evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("LLM et embeddings OpenAI initialisés.")

    # 2. Définition des métriques à calculer
    metrics_to_evaluate = [
        faithfulness,       # Génération: fidèle au contexte ?
        answer_relevancy,   # Génération: réponse pertinente à la question ?
        context_precision,  # Récupération: contexte précis (peu de bruit) ?
        context_recall,     # Récupération: infos clés récupérées (nécessite ground_truth) ?
    ]
    print(f"Métriques sélectionnées: {[m.name for m in metrics_to_evaluate]}")

    # 3. Lancement de l'évaluation Ragas
    print("\nLancement de l'évaluation Ragas (peut prendre du temps)...")
    results = evaluate(
        dataset=evaluation_dataset,
        metrics=metrics_to_evaluate,
        llm=evaluator_llm,
        embeddings=openai_embeddings,
    )
    print("\n--- Évaluation Ragas terminée ---")

    # 4. Affichage des résultats sous forme de DataFrame
    print("\n--- Résultats de l'évaluation (DataFrame) ---")
    results_df = results.to_pandas()
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.max_colwidth', 150) # Ajustez si nécessaire
    print(results_df)

    # 5. Calcul et affichage des scores moyens
    print("\n--- Scores Moyens (sur tout le dataset) ---")
    average_scores = results_df.mean(numeric_only=True)
    print(average_scores)
except Exception as e:
    print(f"\n❌ ERREUR lors de l'initialisation ou de l'évaluation Ragas : {e}")
    print("\nTraceback:")
    traceback.print_exc()

Initialisation LLM et Embeddings OpenAI...
LLM et embeddings OpenAI initialisés.
Métriques sélectionnées: ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']

Lancement de l'évaluation Ragas (peut prendre du temps)...


Evaluating:   0%|          | 0/28 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.



--- Évaluation Ragas terminée ---

--- Résultats de l'évaluation (DataFrame) ---
                                                                                        user_input                                                                                                                                     retrieved_contexts                                                                                                                                               response                                                                                                                                              reference  faithfulness  answer_relevancy  context_precision  context_recall
0                         Y a-t-il un atelier pour découvrir les métiers du numérique à Toulouse ?  [Unnamed: 0: 519\nuid: 65467171\nslug: atelier-decouverte-du-numerique-8834588\ncanonicalurl: https://openagenda.com/francetravail/events/atelier-...  Oui, il y a un atelier intitulé "Atelier découve

In [19]:
results_df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,Y a-t-il un atelier pour découvrir les métiers du numérique à Toulouse ?,[Unnamed: 0: 519\nuid: 65467171\nslug: atelier-decouverte-du-numerique-8834588\ncanonicalurl: https://openagenda.com/francetravail/events/atelier-...,"Oui, il y a un atelier intitulé ""Atelier découverte du numérique"" qui se tiendra le lundi 27 octobre 2025, de 14h00 à 16h00. Cet atelier permettra...",L'événement « ATELIER DECOUVERTE DU NUMERIQUE » propose de s'orienter vers les métiers du numérique. Il a lieu à l'agence France Travail Toulouse ...,0.916667,0.698028,1.000000,0.5
1,Où se déroule le rendez-vous avec l'agence YES pour l'emploi industrie et logistique ?,[Unnamed: 0: 1588\nuid: 80138276\nslug: job-dating-industrie-aeronautique-avec-interval-prestation\ncanonicalurl: https://openagenda.com/francetra...,Je n'ai pas d'informations concernant un rendez-vous avec l'agence YES pour l'emploi industrie et logistique dans ma base de connaissances. \n\nCe...,Le rendez-vous est prévu le mardi 23 juin de 11h à 12h à l'agence France Travail de Borderouge à Toulouse (31200). L'agence YES présente des missi...,0.666667,0.000000,1.000000,0.0
2,Est-ce que tu peux me parler de l'évenement de Job Dating aux Minimes avec Liebherr Aérospace ?,[Unnamed: 0: 1588\nuid: 80138276\nslug: job-dating-industrie-aeronautique-avec-interval-prestation\ncanonicalurl: https://openagenda.com/francetra...,"Je suis désolé, mais je n'ai pas d'informations concernant un événement de Job Dating aux Minimes avec Liebherr Aérospace dans ma base de connaiss...","C'est un job dating le mardi 9 juin à la Maison Toulouse Services Minimes, 4 place du Marché aux Cochons, 31200 Toulouse. Liebherr Aérospace y pro...",0.666667,0.000000,0.000000,0.0
3,Existe-t-il un atelier sur la création d'entreprise à Toulouse ?,[Unnamed: 0: 249\nuid: 17484997\nslug: les-10-questions-que-se-posent-les-createurs-avant-de-se-lancer\ncanonicalurl: https://openagenda.com/franc...,"Oui, un atelier intitulé ""Les 10 questions que se posent les créateurs avant de se lancer"" est prévu à Toulouse. Cet atelier aura lieu le jeudi 24...",L'atelier « De l'idée à l'action : Lancez-vous dans la création d'entreprise ! » s'adresse à ceux qui souhaitent créer une entreprise ou une activ...,0.875000,0.648627,1.000000,0.0
4,Est-ce que tu peux me donner des détails sur le concert de Fazil Say aux Jacobins ?,[Unnamed: 0: 4628\nuid: 8832206\nslug: preambule-du-festival-de-rue-de-ramonville-2026-a-toulouse\ncanonicalurl: https://openagenda.com/mervilla/e...,"Bien sûr, voici les détails concernant le concert de Fazil Say aux Jacobins :\n\n**Titre de l'événement :** Fazil Say - 46E Festival International...","Fazil Say propose un récital dans le cadre du 46e Festival International Piano aux Jacobins à Toulouse. Le programme inclut des œuvres de Bach, Me...",0.800000,0.935661,0.333333,0.5
5,Quand l'agence KIWI Indus/Logistique reçoit-elle à Borderouge ?,[Unnamed: 0: 34\nuid: 82079065\nslug: venez-rencontrer-lagence-demploi-kiwi-induslogistique-pour-trouver-un-emploi-qui-vous-ressemble\ncanonicalur...,"L'agence KIWI Indus/Logistique sera présente à l'agence de Toulouse Borderouge le **Jeudi 25 septembre 2025**, de **9h à 12h**. \n\nPour plus de d...",L'agence KIWI Indus/Logistique est présente à l'agence de Toulouse Borderouge le jeudi 25 septembre 2025 de 9h à 12h. Des postes sont recherchés (...,1.000000,0.784947,1.000000,1.0
6,Récital de Elisabeth Brauss au festival piano toulousain des Jacobins ?,[Unnamed: 0: 55\nuid: 36234745\nslug: elisabeth-brauss-46e-festival-international-piano-aux-jacobins\ncanonicalurl: https://openagenda.com/unidive...,Le récital d'Elisabeth Brauß aura lieu au 46e Festival International Piano aux Jacobins le mardi 16 septembre 2025 à 20h00. Cet événement se dérou...,Elisabeth BRAUß se produit dans le cadre du 46e Festival International Piano aux Jacobins. Il s'agit d'un récital de piano avec 